# Detecção de fraudes em transações de cartão
Projeto DIO: análise de dados desbalanceados, comparação de modelos e explicabilidade.

**Dataset:** `creditcard.csv` (dataset Credit Card Fraud Detection). Defina `DATASET_URL` com o link direto apresentado na Aula 1 ou coloque o CSV localmente. Não versione o CSV no GitHub.

In [ ]:
# Execute uma vez se necessário:
# %pip install -q pandas numpy scikit-learn matplotlib xgboost shap requests

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, precision_recall_fscore_support,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
RANDOM_STATE = 42

## 1. Carregamento e exploração

In [ ]:
DATASET_URL = os.getenv("DATASET_URL", "")  # Link direto do CSV informado na Aula 1
DATASET_PATH = Path("creditcard.csv")
if DATASET_URL:
    df = pd.read_csv(DATASET_URL)
elif DATASET_PATH.exists():
    df = pd.read_csv(DATASET_PATH)
else:
    raise FileNotFoundError("Defina DATASET_URL com o link da Aula 1 ou coloque creditcard.csv nesta pasta.")
required = {"Time", "Amount", "Class"} | {f"V{i}" for i in range(1, 29)}
assert required.issubset(df.columns), f"Colunas ausentes: {required - set(df.columns)}"
assert not df[list(required)].isna().any().any(), "Trate valores ausentes antes do treinamento"
assert set(df.Class.unique()).issubset({0, 1}), "Class deve conter apenas 0 e 1"
print("Dimensões:", df.shape)
print("Distribuição de classes (%):")
print((df.Class.value_counts(normalize=True).sort_index()*100).round(4))
display(df.head())

## 2. Engenharia de atributos e divisão estratificada
A transformação logarítmica de `Amount` é calculada linha a linha, sem vazamento de informação. O `StandardScaler` é ajustado apenas nos dados de treino pelo pipeline.

In [ ]:
X = df.drop(columns="Class").copy()
X["Amount_log"] = np.log1p(X["Amount"].clip(lower=0))
y = df["Class"].astype(int)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=RANDOM_STATE)
print("Treino, validação, teste:", len(X_train), len(X_val), len(X_test))

## 3. Treinar modelos
Pesos de classes ajudam a combater o desbalanceamento. O conjunto de teste permanece intocado até a avaliação final.

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
models = {
    "Regressão Logística": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=14,
        min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=180, max_depth=4, learning_rate=0.07,
        subsample=0.85, colsample_bytree=0.85, scale_pos_weight=scale_pos_weight,
        eval_metric="logloss", tree_method="hist", n_jobs=4, random_state=RANDOM_STATE)
}
for name, model in models.items():
    print("Treinando:", name)
    model.fit(X_train, y_train)

## 4. Escolha do limiar na validação
O limiar é selecionado para maximizar F1 entre aqueles com recall de fraude ≥ 0,80, quando possível. Caso contrário, maximiza F1. O valor 0,80 é um objetivo de exemplo, não uma exigência operacional.

In [ ]:
def choose_threshold(y_true, probabilities, recall_target=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    eligible = np.where(recall[:-1] >= recall_target)[0]
    candidates = eligible if len(eligible) else np.arange(len(thresholds))
    idx = candidates[np.argmax(f1[candidates])]
    return float(thresholds[idx])

thresholds = {}
for name, model in models.items():
    probabilities = model.predict_proba(X_val)[:, 1]
    thresholds[name] = choose_threshold(y_val, probabilities)
print("Limiares escolhidos na validação:", thresholds)

## 5. Avaliação final no teste

In [ ]:
rows = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, model in models.items():
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= thresholds[name]).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, labels=[1], zero_division=0)
    rows.append({"modelo": name, "limiar": thresholds[name], "precisao_fraude": precision[0],
                 "recall_fraude": recall[0], "f1_fraude": f1[0],
                 "roc_auc": roc_auc_score(y_test, probabilities),
                 "average_precision": average_precision_score(y_test, probabilities)})
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    pr, rc, _ = precision_recall_curve(y_test, probabilities)
    axes[0].plot(fpr, tpr, label=name)
    axes[1].plot(rc, pr, label=name)
results = pd.DataFrame(rows).sort_values(["recall_fraude", "f1_fraude"], ascending=False)
display(results.round(4))
axes[0].plot([0,1],[0,1], '--', color='gray')
axes[0].set(title='Curva ROC', xlabel='Taxa de falsos positivos', ylabel='Recall')
axes[1].set(title='Precisão × Recall', xlabel='Recall', ylabel='Precisão')
for ax in axes: ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

In [ ]:
best_name = results.iloc[0]["modelo"]
best_model = models[best_name]
best_probs = best_model.predict_proba(X_test)[:, 1]
best_pred = (best_probs >= thresholds[best_name]).astype(int)
print("Melhor modelo por recall de fraude no teste:", best_name)
print(classification_report(y_test, best_pred, digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=["Normal", "Fraude"], cmap="Blues")
plt.show()

## 6. Importância das variáveis e SHAP
SHAP pode ser custoso. Usamos uma amostra pequena e explicamos o Random Forest ou XGBoost; caso o melhor modelo seja regressão logística, usamos o XGBoost como exemplo explicativo. A explicação não é causal.

In [ ]:
import shap
explain_name = best_name if best_name != "Regressão Logística" else "XGBoost"
explain_model = models[explain_name]
importance = pd.Series(explain_model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(12).to_frame("importancia"))
importance.head(12).sort_values().plot.barh(figsize=(8,5), title=f"Importância: {explain_name}")
plt.tight_layout(); plt.show()

sample = X_test.sample(n=min(100, len(X_test)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(explain_model)
shap_values = explainer(sample)
# Alguns classificadores retornam valores para ambas as classes.
if len(shap_values.values.shape) == 3:
    shap_values = shap_values[:, :, 1]
shap.plots.beeswarm(shap_values, max_display=12, show=True)
shap.plots.waterfall(shap_values[0], max_display=12, show=True)

## 7. Interpretação
Preencha após executar com dados reais: qual modelo apresentou maior recall? Quantos falsos positivos? O limiar escolhido oferece precisão operacional aceitável? Quais variáveis tiveram maior influência no SHAP?

**Limitação:** transações consecutivas ou múltiplas transações do mesmo cartão podem gerar dependência temporal. Esta divisão aleatória estratificada é um baseline; para produção, valide temporalmente, considere custo de investigação e monitore deriva de dados.